# Model 2: BP → ECG Generation
**Dataset**: Brugada-HUCA (PhysioNet)

This notebook trains a **Conditional Variational Autoencoder (CVAE)** that generates realistic 12-lead ECG waveforms conditioned on Blood Pressure values (SBP, DBP).

Since the Brugada-HUCA dataset does not contain BP directly, we:
1. Use the ECG recordings as the target waveform distribution.
2. Synthesize BP conditioning values from a realistic physiological distribution (or inject them from VitalDB statistics during combined training in Model 3).

**Architecture**: Conditional VAE (encoder + decoder with 1D CNN)

**Input**: [SBP, DBP] + noise latent z

**Output**: 12-lead ECG (1200 samples × 12 leads @ 100 Hz)

In [ ]:
! pip install wfdb torch torchvision torchaudio scikit-learn matplotlib numpy pandas tqdm joblib

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import wfdb
from tqdm import tqdm
import warnings
import joblib

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Load Brugada-HUCA Dataset

In [ ]:
# ─── UPDATE THIS PATH to your local dataset folder ────────────────────────────
DATASET_ROOT = r'D:\BPI-Net\brugada-huca-12-lead-ecg-recordings-for-the-study-of-brugada-syndrome-1.0.0\brugada-huca-12-lead-ecg-recordings-for-the-study-of-brugada-syndrome-1.0.0'
DATA_DIR     = os.path.join(DATASET_ROOT, 'files')
META_PATH    = os.path.join(DATASET_ROOT, 'metadata.csv')

# ─── Config ───────────────────────────────────────────────────────────────────
FS           = 100       # 100 Hz per dataset spec
DURATION_SEC = 12        # 12-second recordings
N_SAMPLES    = FS * DURATION_SEC   # 1200 samples per lead
N_LEADS      = 12
LATENT_DIM   = 64        # VAE latent dimension
BP_DIM       = 2         # [SBP_norm, DBP_norm]

In [ ]:
def load_brugada_dataset(data_dir, meta_path):
    """
    Load all ECG recordings from the Brugada-HUCA dataset.
    Returns:
        ecg_array   : (N, 12, 1200) float32
        labels      : (N,) int  [0=Normal, 1=Brugada]
        patient_ids : list of str
    """
    metadata = pd.read_csv(meta_path)
    print(f'Metadata loaded: {len(metadata)} records')
    print(metadata['brugada'].value_counts().rename({0: 'Normal', 1: 'Brugada'}))

    ecg_list    = []
    label_list  = []
    pid_list    = []

    for _, row in tqdm(metadata.iterrows(), total=len(metadata), desc='Loading ECGs'):
        pid = str(int(row['patient_id']))
        rec_path = os.path.join(data_dir, pid, pid)

        try:
            record = wfdb.rdrecord(rec_path)
            sig = record.p_signal  # (1200, 12)

            if sig is None or sig.shape != (N_SAMPLES, N_LEADS):
                continue
            if np.any(np.isnan(sig)) or np.any(np.isinf(sig)):
                continue

            ecg_list.append(sig.T.astype(np.float32))  # → (12, 1200)
            label_list.append(int(row['brugada']))
            pid_list.append(pid)

        except Exception as e:
            continue

    print(f'\nSuccessfully loaded: {len(ecg_list)} recordings')
    return np.array(ecg_list), np.array(label_list), pid_list


ecg_data, labels, patient_ids = load_brugada_dataset(DATA_DIR, META_PATH)
print(f'ECG array shape: {ecg_data.shape}')   # (N, 12, 1200)

## 2. Preprocessing & BP Conditioning

In [ ]:
# ─── Normalize ECG per-lead, per-recording ────────────────────────────────────
def normalize_ecg(ecg):   # (N, 12, 1200)
    mu  = ecg.mean(axis=2, keepdims=True)
    std = ecg.std(axis=2,  keepdims=True) + 1e-8
    return (ecg - mu) / std

ecg_norm = normalize_ecg(ecg_data)
print(f'ECG normalized shape: {ecg_norm.shape}')

# ─── Synthesize BP conditioning values ────────────────────────────────────────
# Since Brugada-HUCA has no BP, we simulate realistic values.
# Normal subjects: SBP~120±15, DBP~80±10
# Brugada subjects: slightly higher variability
np.random.seed(SEED)
N = len(ecg_norm)

sbp_syn = np.where(
    labels == 0,
    np.random.normal(120, 15, N),   # Normal
    np.random.normal(125, 18, N)    # Brugada – slightly elevated
).clip(70, 200).astype(np.float32)

dbp_syn = np.where(
    labels == 0,
    np.random.normal(80, 10, N),
    np.random.normal(82, 12, N)
).clip(40, 130).astype(np.float32)

# Normalize BP to [-1, 1] using fixed physiological range
def norm_bp(v, lo, hi):
    return ((v - lo) / (hi - lo) * 2 - 1).astype(np.float32)

sbp_norm_vals = norm_bp(sbp_syn, 70, 200)
dbp_norm_vals = norm_bp(dbp_syn, 40, 130)
bp_cond = np.stack([sbp_norm_vals, dbp_norm_vals], axis=1)  # (N, 2)

print(f'BP conditioning shape: {bp_cond.shape}')

# Save the normalization params for Model 3
bp_bounds = {'sbp': (70, 200), 'dbp': (40, 130)}
joblib.dump(bp_bounds, 'bp_bounds.pkl')

In [ ]:
# ─── Visualize sample ECGs ─────────────────────────────────────────────────────
LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
t = np.arange(N_SAMPLES) / FS

fig, axes = plt.subplots(4, 3, figsize=(16, 10))
fig.suptitle(f'Sample ECG – Patient {patient_ids[0]} (Label: {"Brugada" if labels[0] else "Normal"})', fontsize=14)
for i, ax in enumerate(axes.flat):
    ax.plot(t, ecg_norm[0, i], linewidth=0.8)
    ax.set_title(LEAD_NAMES[i])
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('mV (norm)')
    ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, ecg, bp_cond, labels):
        self.ecg    = torch.tensor(ecg,     dtype=torch.float32)   # (N,12,1200)
        self.bp     = torch.tensor(bp_cond, dtype=torch.float32)   # (N,2)
        self.labels = torch.tensor(labels,  dtype=torch.long)      # (N,)

    def __len__(self):
        return len(self.ecg)

    def __getitem__(self, idx):
        return self.ecg[idx], self.bp[idx], self.labels[idx]


dataset  = ECGDataset(ecg_norm, bp_cond, labels)
n_total  = len(dataset)
n_train  = int(0.70 * n_total)
n_val    = int(0.15 * n_total)
n_test   = n_total - n_train - n_val

train_set, val_set, test_set = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')

## 3. Model Architecture: Conditional VAE

In [ ]:
class ConvBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, stride=2, padding=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=padding),
            nn.BatchNorm1d(out_ch),
            nn.GELU()
        )
    def forward(self, x):
        return self.net(x)


class ConvTransBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, stride=2, padding=3, out_padding=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose1d(in_ch, out_ch, kernel, stride=stride,
                               padding=padding, output_padding=out_padding),
            nn.BatchNorm1d(out_ch),
            nn.GELU()
        )
    def forward(self, x):
        return self.net(x)


class CVAEEncoder(nn.Module):
    """
    Encodes ECG + BP condition into latent (mu, log_var).
    Input: ecg (B,12,1200), bp (B,2)
    """
    def __init__(self, latent_dim=LATENT_DIM, bp_dim=BP_DIM):
        super().__init__()
        # BP embedding (broadcast as extra channels)
        self.bp_embed = nn.Linear(bp_dim, 4 * 1200)

        # CNN on concatenated ECG+BP
        self.cnn = nn.Sequential(
            ConvBlock1D(12 + 4, 32,  kernel=15, stride=2, padding=7),   # 600
            ConvBlock1D(32,     64,  kernel=9,  stride=2, padding=4),   # 300
            ConvBlock1D(64,     128, kernel=7,  stride=2, padding=3),   # 150
            ConvBlock1D(128,    256, kernel=5,  stride=2, padding=2),   # 75
            ConvBlock1D(256,    512, kernel=5,  stride=3, padding=2),   # 25
        )
        self.flatten_dim = 512 * 25
        self.fc_mu      = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_log_var = nn.Linear(self.flatten_dim, latent_dim)

    def forward(self, ecg, bp):
        # bp_embed → (B, 4, 1200) to concat with ECG channels
        bp_feat = self.bp_embed(bp).view(bp.size(0), 4, 1200)
        x = torch.cat([ecg, bp_feat], dim=1)   # (B, 16, 1200)
        x = self.cnn(x).view(x.size(0), -1)
        return self.fc_mu(x), self.fc_log_var(x)


class CVAEDecoder(nn.Module):
    """
    Decodes latent z + BP condition into 12-lead ECG.
    """
    def __init__(self, latent_dim=LATENT_DIM, bp_dim=BP_DIM):
        super().__init__()
        self.fc = nn.Linear(latent_dim + bp_dim, 512 * 25)

        self.deconv = nn.Sequential(
            ConvTransBlock1D(512, 256, kernel=5, stride=3, padding=2, out_padding=2),  # 75
            ConvTransBlock1D(256, 128, kernel=5, stride=2, padding=2, out_padding=1),  # 150
            ConvTransBlock1D(128,  64, kernel=7, stride=2, padding=3, out_padding=1),  # 300
            ConvTransBlock1D( 64,  32, kernel=9, stride=2, padding=4, out_padding=1),  # 600
            ConvTransBlock1D( 32,  12, kernel=15, stride=2, padding=7, out_padding=1), # 1200
        )
        self.out_act = nn.Tanh()   # ECG is roughly in [-3, 3] mV norm

    def forward(self, z, bp):
        x = torch.cat([z, bp], dim=1)   # (B, latent+2)
        x = self.fc(x).view(x.size(0), 512, 25)
        x = self.deconv(x)
        return self.out_act(x)          # (B, 12, 1200)


class CVAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, bp_dim=BP_DIM):
        super().__init__()
        self.encoder  = CVAEEncoder(latent_dim, bp_dim)
        self.decoder  = CVAEDecoder(latent_dim, bp_dim)
        self.latent_dim = latent_dim

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, ecg, bp):
        mu, log_var = self.encoder(ecg, bp)
        z           = self.reparameterize(mu, log_var)
        recon       = self.decoder(z, bp)
        return recon, mu, log_var

    def generate(self, bp, n_samples=1):
        """Generate ECGs from BP values only (no encoder needed)."""
        z = torch.randn(n_samples, self.latent_dim).to(bp.device)
        if bp.shape[0] == 1 and n_samples > 1:
            bp = bp.expand(n_samples, -1)
        return self.decoder(z, bp)


model = CVAE().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CVAE parameters: {total_params:,}')

## 4. Training

In [ ]:
def vae_loss(recon, target, mu, log_var, beta=0.001):
    """
    ELBO = Reconstruction loss + beta * KL divergence.
    Beta-VAE: smaller beta keeps generation quality high.
    """
    recon_loss = F.mse_loss(recon, target, reduction='mean')
    kl_loss    = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
    return recon_loss + beta * kl_loss, recon_loss.item(), kl_loss.item()


EPOCHS   = 100
LR       = 3e-4
PATIENCE = 15
BETA     = 0.001

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = total_recon = total_kl = 0
    with torch.set_grad_enabled(train):
        for ecg, bp, _ in loader:
            ecg, bp = ecg.to(device), bp.to(device)
            recon, mu, log_var = model(ecg, bp)
            loss, r, k = vae_loss(recon, ecg, mu, log_var, beta=BETA)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            bs = ecg.size(0)
            total_loss  += loss.item()  * bs
            total_recon += r * bs
            total_kl    += k * bs
    n = len(loader.dataset)
    return total_loss/n, total_recon/n, total_kl/n


train_losses, val_losses = [], []
best_val = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_r, tr_k = run_epoch(train_loader, train=True)
    vl_loss, vl_r, vl_k = run_epoch(val_loader,   train=False)
    scheduler.step(vl_loss)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)

    if vl_loss < best_val:
        best_val = vl_loss
        torch.save(model.state_dict(), 'best_bp_to_ecg.pth')
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0 or patience_counter == 0:
        print(f'Epoch {epoch:3d} | Train {tr_loss:.4f} (R:{tr_r:.4f} KL:{tr_k:.4f}) '
              f'| Val {vl_loss:.4f} (R:{vl_r:.4f} KL:{vl_k:.4f})')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'Best val loss: {best_val:.4f}')

## 5. Evaluation & Visualization

In [ ]:
# ─── Loss curves ──────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Validation')
plt.xlabel('Epoch')
plt.ylabel('ELBO Loss')
plt.title('Training Curves – BP → ECG (CVAE)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ─── Reconstruct vs Real ECG ──────────────────────────────────────────────────
model.load_state_dict(torch.load('best_bp_to_ecg.pth', map_location=device))
model.eval()

LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
t = np.arange(N_SAMPLES) / FS

# Get one test sample
test_ecg, test_bp, test_label = test_set[0]
test_ecg = test_ecg.unsqueeze(0).to(device)
test_bp  = test_bp.unsqueeze(0).to(device)

with torch.no_grad():
    recon_ecg, _, _ = model(test_ecg, test_bp)

real = test_ecg.squeeze().cpu().numpy()
recon = recon_ecg.squeeze().cpu().numpy()

fig, axes = plt.subplots(12, 2, figsize=(16, 24))
fig.suptitle('Real vs Reconstructed ECG (12 leads)', fontsize=14)

for i in range(12):
    axes[i, 0].plot(t, real[i],  color='blue',   linewidth=0.7)
    axes[i, 0].set_title(f'{LEAD_NAMES[i]} – Real')
    axes[i, 0].set_ylabel('mV (norm)')
    axes[i, 0].grid(True, alpha=0.3)

    axes[i, 1].plot(t, recon[i], color='orange', linewidth=0.7)
    axes[i, 1].set_title(f'{LEAD_NAMES[i]} – Reconstructed')
    axes[i, 1].set_ylabel('mV (norm)')
    axes[i, 1].grid(True, alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

In [ ]:
# ─── Generate new ECGs from BP values ─────────────────────────────────────────
def generate_ecg_from_bp(sbp_mmhg, dbp_mmhg, n_samples=3):
    """Generate ECG waveforms given SBP and DBP in mmHg."""
    bounds = joblib.load('bp_bounds.pkl')
    sbp_n  = (sbp_mmhg - bounds['sbp'][0]) / (bounds['sbp'][1] - bounds['sbp'][0]) * 2 - 1
    dbp_n  = (dbp_mmhg - bounds['dbp'][0]) / (bounds['dbp'][1] - bounds['dbp'][0]) * 2 - 1
    bp_t   = torch.tensor([[sbp_n, dbp_n]], dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        ecgs = model.generate(bp_t, n_samples=n_samples)

    return ecgs.cpu().numpy()  # (n_samples, 12, 1200)


# ─── Compare ECGs at different BP levels ──────────────────────────────────────
bp_scenarios = [
    (120, 80,  'Normal BP (120/80)'),
    (160, 100, 'Hypertension (160/100)'),
    (90,  60,  'Hypotension (90/60)'),
]

fig, axes = plt.subplots(len(bp_scenarios), 1, figsize=(14, 10))
t = np.arange(N_SAMPLES) / FS

for ax, (sbp, dbp, title) in zip(axes, bp_scenarios):
    gen = generate_ecg_from_bp(sbp, dbp, n_samples=1)[0]  # (12, 1200)
    ax.plot(t, gen[0], label='Lead I',  linewidth=0.8)
    ax.plot(t, gen[6] - 3, label='V1 (offset)', linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel('mV (norm)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Generated ECG Lead I at Different BP Levels', fontsize=14)
plt.tight_layout()
plt.show()

print('Model saved as: best_bp_to_ecg.pth')

In [ ]:
# ─── Reconstruction MSE on test set ───────────────────────────────────────────
model.eval()
test_mse_list = []

with torch.no_grad():
    for ecg, bp, _ in test_loader:
        ecg, bp = ecg.to(device), bp.to(device)
        recon, _, _ = model(ecg, bp)
        mse = F.mse_loss(recon, ecg, reduction='none').mean(dim=[1, 2])
        test_mse_list.extend(mse.cpu().numpy())

test_mse_arr = np.array(test_mse_list)
print(f'Test Reconstruction MSE: {test_mse_arr.mean():.4f} ± {test_mse_arr.std():.4f}')

plt.figure(figsize=(8, 4))
plt.hist(test_mse_arr, bins=30, edgecolor='k', alpha=0.7)
plt.xlabel('Per-sample MSE')
plt.ylabel('Count')
plt.title('Distribution of Reconstruction Errors on Test Set')
plt.grid(True, alpha=0.3)
plt.show()